In [1]:
from google.colab import drive, userdata

drive.mount('/content/drive')
%cd /content/drive/MyDrive/project-lumen/

token = userdata.get('GHToken')

!git remote set-url origin https://{token}@github.com/RafaelWhoa/project-lumen.git
!git pull

Mounted at /content/drive
/content/drive/MyDrive/project-lumen
Already up to date.


In [2]:
!pip install -q torch torchvision transformers scikit-learn matplotlib

In [4]:
!mkdir -p /content/data/nih-chest-xrays
!unzip -q /content/drive/MyDrive/Datasets/nih-chest-xrays.zip -d /content/data/nih-chest-xrays/

In [5]:
path = "/content/data/nih-chest-xrays/"

In [6]:
import os

for root, dirs, files in os.walk(path):
    nivel = root.replace(path, '').count(os.sep)
    indent = ' ' * 2 * nivel
    print(f'{indent}{os.path.basename(root)}/')
    if nivel < 2:  # não lista todos os arquivos, só a estrutura
        subindent = ' ' * 2 * (nivel + 1)
        for f in files[:3]:
            print(f'{subindent}{f}')

/
  ARXIV_V5_CHESTXRAY.pdf
  README_CHESTXRAY.pdf
  FAQ_CHESTXRAY.pdf
images_006/
  images/
    00013003_021.png
    00011850_008.png
    00013539_003.png
images_004/
  images/
    00006978_001.png
    00007780_008.png
    00009209_000.png
images_002/
  images/
    00002275_006.png
    00001385_013.png
    00001876_000.png
images_008/
  images/
    00016496_001.png
    00016896_007.png
    00017439_000.png
images_011/
  images/
    00026275_000.png
    00026270_000.png
    00025460_000.png
images_007/
  images/
    00015530_059.png
    00014786_000.png
    00014472_006.png
images_009/
  images/
    00018972_030.png
    00018921_000.png
    00020204_021.png
images_005/
  images/
    00010800_000.png
    00010481_009.png
    00011034_024.png
images_003/
  images/
    00005626_005.png
    00004275_000.png
    00004577_008.png
images_001/
  images/
    00000356_000.png
    00000359_002.png
    00001019_002.png
images_012/
  images/
    00029407_000.png
    00029102_001.png
    00030608_003

In [7]:
import os
import glob

# ajusta pro caminho onde está o dataset baixado
base_path = path

# pega todas as imagens de todas as pastas images_XXX/images/
all_image_paths = glob.glob(os.path.join(base_path, "images_*", "images", "*.png"))

print("Total de imagens encontradas:", len(all_image_paths))

# mapa nome -> caminho completo (o nome do arquivo é único no dataset todo)
image_path_map = {os.path.basename(p): p for p in all_image_paths}

Total de imagens encontradas: 112120


In [8]:
import pandas as pd

csv_path = os.path.join(base_path, "Data_Entry_2017.csv")
df = pd.read_csv(csv_path)

print(df.shape)
df.head()

(112120, 12)


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN


In [9]:
train_val_list_path = os.path.join(base_path, "train_val_list.txt")
test_list_path = os.path.join(base_path, "test_list.txt")

with open(train_val_list_path) as f:
    train_val_files = set(f.read().splitlines())

with open(test_list_path) as f:
    test_files = set(f.read().splitlines())

print("Train+Val:", len(train_val_files), "| Test:", len(test_files))

Train+Val: 86524 | Test: 25596


In [10]:
# lista oficial das 14 patologias do NIH (sem contar "No Finding")
disease_labels = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass',
    'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema',
    'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia'
]

# cria uma coluna binária pra cada doença
for label in disease_labels:
    df[label] = df['Finding Labels'].apply(lambda x: 1 if label in x.split('|') else 0)

# confere: "No Finding" deve resultar em vetor todo zero
df[disease_labels].sum().sort_values(ascending=False)

,0
Infiltration,19894
Effusion,13317
Atelectasis,11559
Nodule,6331
Mass,5782
Pneumothorax,5302
Consolidation,4667
Pleural_Thickening,3385
Cardiomegaly,2776
Emphysema,2516


In [11]:
from sklearn.model_selection import GroupShuffleSplit

df_trainval = df[df['Image Index'].isin(train_val_files)].copy()
df_test     = df[df['Image Index'].isin(test_files)].copy()

# split 85/15 respeitando Patient ID (mesmo paciente não pode cair em train e val ao mesmo tempo)
splitter = GroupShuffleSplit(test_size=0.15, n_splits=1, random_state=42)
train_idx, val_idx = next(splitter.split(df_trainval, groups=df_trainval['Patient ID']))

df_train = df_trainval.iloc[train_idx]
df_val   = df_trainval.iloc[val_idx]

print("Treino:", len(df_train), "| Val:", len(df_val), "| Teste:", len(df_test))

Treino: 73916 | Val: 12608 | Teste: 25596


In [12]:
import torch
from torch.utils.data import Dataset
from PIL import Image

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, image_path_map, label_cols, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_path_map = image_path_map
        self.label_cols = label_cols
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.image_path_map[row['Image Index']]
        image = Image.open(img_path).convert('RGB')  # NIH vem em grayscale, convertendo pra 3 canais (padrão dos modelos pré-treinados)

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(row[self.label_cols].values.astype('float32'))
        return image, labels

In [13]:
import torch

# peso inversamente proporcional à frequência de cada doença
pos_counts = df_train[disease_labels].sum().values
neg_counts = len(df_train) - pos_counts
pos_weight = torch.tensor(neg_counts / pos_counts, dtype=torch.float32)

print(pos_weight)

tensor([  9.5549,  49.8713,   8.9684,   5.2361,  20.4125,  17.2870,  98.6172,
         31.3201,  28.7808,  60.5454,  57.0188,  69.3292,  38.5273, 653.1239])


In [14]:
from sklearn.metrics import roc_auc_score

In [20]:
from torchvision import transforms
from torch.utils.data import DataLoader

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225]),
])

train_dataset = ChestXrayDataset(df_train, image_path_map, disease_labels, transform=transform_train)
val_dataset   = ChestXrayDataset(df_val, image_path_map, disease_labels, transform=transform_eval)
test_dataset  = ChestXrayDataset(df_test, image_path_map, disease_labels, transform=transform_eval)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=4, pin_memory=True)

In [24]:
import torch.nn as nn
from torchvision import models

model = models.resnet50(weights='IMAGENET1K_V2')

# troca a última camada: 14 saídas (uma por doença), sem softmax
# porque cada doença é independente (multi-rótulo, não multi-classe)
model.fc = nn.Linear(model.fc.in_features, len(disease_labels))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

In [28]:
scaler = torch.cuda.amp.GradScaler()

def train_one_epoch(model, loader, criterion, optimizer, device, scaler):
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)

/tmp/ipykernel_4428/48312462.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [26]:
from sklearn.metrics import roc_auc_score
import numpy as np

def evaluate(model, loader, criterion, device, label_names):
    model.eval()
    running_loss = 0.0
    all_labels = []
    all_preds = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)

            all_labels.append(labels.cpu().numpy())
            all_preds.append(torch.sigmoid(outputs).cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)

    # AUC por classe (pula classes que não têm nenhum positivo no batch de validação, senão dá erro)
    aucs = {}
    for i, name in enumerate(label_names):
        if len(np.unique(all_labels[:, i])) > 1:
            aucs[name] = roc_auc_score(all_labels[:, i], all_preds[:, i])
        else:
            aucs[name] = float('nan')

    mean_auc = np.nanmean(list(aucs.values()))
    return epoch_loss, aucs, mean_auc

In [29]:
import time

num_epochs = 5

history = []

for epoch in range(num_epochs):
    start = time.time()

    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device, scaler)
    val_loss, val_aucs, val_mean_auc = evaluate(model, val_loader, criterion, device, disease_labels)

    elapsed = time.time() - start

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val Mean AUC: {val_mean_auc:.4f} | Tempo: {elapsed/60:.1f} min")

    history.append({
        "epoch": epoch+1, "train_loss": train_loss,
        "val_loss": val_loss, "val_mean_auc": val_mean_auc
    })

    # salva checkpoint a cada época (importante: sessão pode cair)
    torch.save(model.state_dict(), f"/content/drive/MyDrive/project-lumen/checkpoint_epoch{epoch+1}.pt")

Epoch 1/5 | Train Loss: 1.1426 | Val Loss: 1.0704 | Val Mean AUC: 0.7908 | Tempo: 7.9 min
Epoch 2/5 | Train Loss: 1.0161 | Val Loss: 1.0268 | Val Mean AUC: 0.8053 | Tempo: 7.8 min
Epoch 3/5 | Train Loss: 0.9470 | Val Loss: 1.0365 | Val Mean AUC: 0.8114 | Tempo: 7.8 min
Epoch 4/5 | Train Loss: 0.8774 | Val Loss: 1.0882 | Val Mean AUC: 0.8225 | Tempo: 7.8 min
Epoch 5/5 | Train Loss: 0.8373 | Val Loss: 1.0340 | Val Mean AUC: 0.8206 | Tempo: 7.8 min
